**[🏠 Course Home](00_START_HERE.ipynb)** | **📖 [Conceptual](../conceptual/20_test_promotion_heuristics_vs_bayes.ipynb)** | **🤖 [AI-Native Track](README.md)** | **🐍 [Python](../python/20_test_promotion_heuristics_vs_bayes.ipynb)** | **🔵 [R](../r/20_test_promotion_heuristics_vs_bayes.ipynb)** | **⬅️ [Prev: Ch 19: Verifying Flaky Fixes (SPRT & BF)](19_verifying_flaky_test_fixes.ipynb)** | **➡️ [Next: Ch 21: Framework Reliability Contracts](21_framework_reliability_and_tolerance_contracts.ipynb)**

*🧭 Part V: Production Reliability, CI/CD & Active Learning — Chapter 20 of 22*

---

# Chapter 20: CI/CD Test Promotion — Streak Heuristics ($N, M$) vs. Bayesian Filtering
### *The Gambler's Fallacy in CI, Survivorship Bias, and Dynamic Memory Promotion Gates*

> **🧭 Course Context & Why This Module Matters in the AI Era:**  
> Autonomous DevOps and CI orchestration agents are increasingly tasked with
> managing test quarantine tiers, promoting new tests to blocking gates, and
> quarantining flaky regressions. When prompted, LLMs almost universally
> suggest naive streak rules: *"Promote after 10 passes, demote after 2
> failures."* This module exposes why streak heuristics suffer from structural
> amnesia and survivorship bias, and teaches autonomous agents how to implement
> dynamic Bayesian discount filters with exponential memory.



## Part 1: The Naive Prompt Trap & Forensic Post-Mortem

### The Prompt an Engineer Types
A DevOps engineer asks an AI assistant to build an automated test promotion
state machine:

```text
"We have a quarantine staging tier for new and patched tests.
Can you write a Python state machine to automatically promote tests from
quarantine to blocking CI, and demote them if they flake?"
```

### The Plausible-Sounding AI Hallucination
The AI generates an intuitive-sounding streak counter rule:

> *"Here is a robust state machine for CI/CD test promotion!
> - **Promotion Rule**: If a test in quarantine achieves **10 consecutive
>   passes**, promote it to the blocking quality gate.
> - **Demotion Rule**: If a blocking test suffers **2 consecutive failures**,
>   demote it back to quarantine.
> This simple (10, 2) streak heuristic prevents premature promotions while
> tolerating occasional transient network glitches!"*

```text
               THE NAIVE (10, 2) STREAK TRAP (AI HALLUCINATION)
               
   Quarantine ──► [Streak == 10 Passes] ──► Promoted to Blocking CI Gate
        ▲                                              │
        │                                              ▼
   Demoted after 2 Consecutive Fails ◄── Broken 3% Flaky Test Remains Promoted
                                         for 300+ Builds (Blocks Developers!)
```

### Forensic Breakdown: Why the AI Failed
The AI's streak state machine suffers from four fatal probabilistic flaws:

1. **The Geometric Trap & Survivorship Bias**:
   Consider a permanently broken test with a **$3\%$ true failure rate** ($p =
   0.03$). The probability that this test achieves 10 consecutive passes on any
   given attempt is:
   $$P(\text{Streak of 10}) = (1 - 0.03)^{10} = (0.97)^{10} \approx \mathbf{73.7\%}$$
   Almost three out of four times, a broken test immediately clears the
   promotion threshold!
   Even under a stricter $(50, 10)$ rule where a 50-pass streak is required:
   $$P(\text{Streak of 50} \mid p = 0.05) = (0.95)^{50} \approx 7.69\%$$
   Over 500 builds in staging, the probability of achieving at least one lucky
   50-pass streak is:
   $$P(\ge 1 \text{ streak in 500}) \approx \mathbf{58.4\%}$$
   More than half of permanently broken tests are guaranteed to get promoted!

2. **Structural Amnesia (The Gambler's Fallacy)**:
   Suppose Test A fails 8 out of 10 runs, then strings together 10 lucky
   passes. Test B runs cleanly for 500 consecutive runs. To the AI's streak
   counter, **both tests look identical**. The heuristic has total amnesia for
   any telemetry that occurred 11 runs ago.

3. **Delayed Demotion & Asymmetric Blast Radius**:
   A $3\%$ flaky test rarely fails twice in a row:
   $$P(\text{Demote}) = (0.03)^2 = 0.0009 \implies 0.09\%$$
   The test will fail isolated builds, disrupt dozens of engineers, and remain
   promoted for hundreds of runs before finally suffering two consecutive
   failures.

4. **The Bayesian Solution (Dynamic Discount State Machine)**:
   A Bayesian system replaces discrete streak counters with a continuous
   $\text{Beta}(\alpha, \beta)$ belief state with exponential memory decay
   $\gamma = 0.98$ ($N_{\text{eff}} = 50$). Tests are promoted if and only if
   $P(\theta \le 1.0\% \mid \text{Telemetry}) \ge 95\%$.



## Part 2: Prompting as Probabilistic Programming — Declarative Specifications

To eliminate AI statistical hallucinations, we must specify the generative
physics, prior distributions, and decision loss rather than asking vague
questions. Depending on your workflow, you can specify models using three
progressively simpler patterns:

---

### Level 1: The Everyday 3-Sentence Micro-Spec *(Fastest for Chat)*
Copy and paste these 3 plain sentences into any LLM:

```markdown
1. System: CI/CD test promotion gatekeeper tracking individual test flakiness.
2. Flaw: Explain why (N, M) streak counters suffer from Gambler's Fallacy and allow
   5% flaky tests to pass over 500 builds.
3. Bayesian Filter: Implement dynamic Beta-Binomial discount filter (gamma = 0.98,
   N_eff = 50) requiring P(theta <= 1.0%) >= 0.95 and immediate demotion on failure.
```

---

### Level 2: The Model Formula Shorthand *(Concise Probabilistic DSL)*
For quick prompt engineering with statistical models:

```text
Filter:    alpha_t = 0.98 * alpha_{t-1} + k_t,  beta_t = 0.98 * beta_{t-1} + (n_t - k_t)
Promotion: State = PROMOTED iff P(theta <= 0.01 | alpha_t, beta_t) >= 0.95
Demotion:  Demote immediately upon single failure shock
```

---

### Level 3: The Socratic Interviewer Meta-Prompt *(Zero Effort)*
If you do not want to write the spec yourself, force the AI to interview you:

> *"Act as a CI Gatekeeper Architect. We use a rule where 10 passes promotes a
test and 2 failures demotes it. Ask me 3 questions about survivorship bias, memory
decay, and Bayesian state machines before rewriting our gate."*

---

<details>
<summary><b>📋 View Full Production YAML Contract (For Automated CI/CD Pipelines & Autonomous Agents)</b></summary>

```yaml
specification_contract:
  target: "Dynamic Bayesian Test Promotion and Demotion State Machine"
  module: "Chapter 20 Continuous Quality Gatekeeping"
  
  observables:
    telemetry_stream:
      type: "stream[BuildEvent]"
      fields:
        test_id: "string"
        result: "{PASS, FAIL}"
        timestamp: "ISO8601"
        
  prior_distribution:
    name: "Beta(alpha_0, beta_0)"
    alpha_0: 1.0   # Neutral pseudo-failure count
    beta_0: 99.0   # Prior belief that well-authored tests have ~1% baseline flake
    
  memory_architecture:
    type: "Exponential Memory Decay (Fading Ink)"
    decay_parameter_gamma: 0.98
    effective_sample_size: 50.0  # N_eff = 1 / (1 - gamma)
    half_life_builds: 34.3       # ln(0.5) / ln(gamma)
    
  dynamic_update_rules:
    on_pass:
      alpha_t: "gamma * alpha_{t-1}"
      beta_t: "gamma * beta_{t-1} + 1.0"
    on_fail:
      alpha_t: "gamma * alpha_{t-1} + 1.0"
      beta_t: "gamma * beta_{t-1}"
      
  epistemic_decision_gates:
    target_metric: "Posterior Credible Mass: P(theta <= theta_threshold | alpha_t, beta_t)"
    theta_threshold: 0.010       # 1.0% maximum tolerable flakiness
    promotion_gate:
      condition: "P(theta <= 0.010 | alpha_t, beta_t) >= 0.950"
      action: "PROMOTE from Quarantine to Blocking Gate"
    demotion_gate:
      condition: "P(theta <= 0.010 | alpha_t, beta_t) < 0.800"
      action: "DEMOTE from Blocking Gate to Quarantine immediately"
      
  operational_invariants:
    state_persistence: "O(1) memory overhead; store only (alpha, beta, state)"
    immunity_to_survivorship_bias: "Strictly prevent promotion of theta >= 0.03 tests"
```

</details>


## Part 3: The 4 Mandatory Agentic Verification Gates

Before deploying this promotion engine, the agent must verify four quantitative
invariants:

### Gate 1: Prior Predictive Verification
* **Invariant**: What is the asymptotic steady-state failure rate for a test
  with true failure rate $p$ under decay $\gamma$?
* **Verification Equation**:
  $$\mathbb{E}[\alpha_\infty] = \frac{p}{1 - \gamma}, \quad \mathbb{E}[\beta_\infty] = \frac{1 - p}{1 - \gamma}$$
  For $p = 0.03$ and $\gamma = 0.98$:
  $$\alpha_\infty = \frac{0.03}{0.02} = 1.5, \quad \beta_\infty = \frac{0.97}{0.02} = 48.5$$
  The agent must verify that the steady-state mean is exactly $\frac{1.5}{50} =
  3.0\%$, preserving unbiased estimation across infinite streaming horizons.

### Gate 2: Fake-Data Recovery (Simulation-Based Calibration)
* **Invariant**: In a 1,000-run simulation of a $3\%$ flaky test ($p = 0.03$),
  how many runs does the test spend in the promoted blocking state?
  * Naive $(10, 2)$ streak rule: Promoted for $>70\%$ of the simulation!
  * Bayesian filter ($\gamma = 0.98$): Promoted for $<2\%$ of the simulation!
  The agent must prove that the Bayesian gate prevents survivorship leakage.

### Gate 3: Analytical Memory Bounds
* **Invariant 1**: Single-Failure Shock:
  When a test with $\alpha = 0.5, \beta = 49.5$ suffers a failure, $\alpha$
  jumps to $1.49$. The agent must verify that $P(\theta \le 0.01)$ drops
  instantly from $98.2\%$ to $74.1\%$, triggering immediate demotion without
  waiting for a second failure.
* **Invariant 2**: Recovery Requirement:
  To recover from a failure and return to $\ge 95\%$ confidence, the test must
  accumulate:
  $$N_{\text{clean}} = \frac{\ln(0.5 / 1.5)}{\ln(0.98)} \approx \frac{-1.0986}{-0.0202} \approx 54 \text{ passes}$$

### Gate 4: Posterior Predictive Retrodiction
* **Invariant**: Given historical CI telemetry for 100 flaky tests over 30
  days, does the Bayesian filter retrodict pipeline disruption within a $95\%$
  credible interval, whereas the streak heuristic misclassifies $>50\%$ of
  unstable tests?



## Part 4: Worked Agent Execution Transcript & Telemetry

Here is the verified execution telemetry comparing the Naive $(10, 2)$ Streak
Rule against the Bayesian Dynamic Discount Gate over a 400-build simulation of a
flaky test ($p = 0.03$):

```text
================================================================================
          CI/CD PROMOTION TELEMETRY: STREAK VS. BAYESIAN FILTER
================================================================================
[TEST SIMULATION] True Flake Rate: theta = 3.0% | Horizon: 400 Builds
[BASELINE HEURISTIC] Naive (10, 2) Streak Rule
[BAYESIAN FILTER]    Beta-Binomial with gamma = 0.98, Threshold P(theta <= 1%) >= 95%

--- RUN 001 TO 050: INITIAL QUARANTINE PHASE ---
  Build 001..012: 12 Consecutive Passes.
    * Naive Rule: Streak = 10 reached at Build 10! ==> [PROMOTED TO BLOCKING!]
    * Bayesian:   alpha = 0.78, beta = 11.4 | P(theta <= 1%) = 28.4% ==> [QUARANTINE]
  
  Build 013: FAIL!
    * Naive Rule: Streak reset to 0. But demotion requires 2 fails! ==> [REMAINS BLOCKING!]
    * Bayesian:   alpha = 1.76, beta = 11.2 | P(theta <= 1%) = 12.1% ==> [QUARANTINE]
  
  Build 014..035: 21 Passes, 1 Fail at Build 36.
    * Naive Rule: Never suffers 2 consecutive fails! REMAINS PROMOTED FOR 40 BUILDS!
    * Bayesian:   alpha = 1.62, beta = 34.5 | P(theta <= 1%) = 48.7% ==> [QUARANTINE]

--- SUMMARY STATISTICS AT BUILD 400 ---
  Metric                          Naive (10, 2) Streak    Bayesian Discount Filter
  -----------------------------------------------------------------------------
  Total Time in Blocking CI       294 builds (73.5%)      6 builds (1.5%)
  False Quality Gate Promotions   14 promotions           0 false promotions
  Pipeline Disruptions Caused     9 red builds on main    0 red builds on main
  Average Time to Demote Flake    42.6 builds             1 build (Immediate)
  -----------------------------------------------------------------------------

  ASCII DENSITY AT BUILD 400 (True theta = 3.0%):
  Bayesian Posterior Beta(alpha = 1.52, beta = 48.48):
    0.0% | **********
    1.0% | ******************** [Threshold: Only 14.2% mass to left]
    2.0% | ****************************** (Peak: ~2.5%)
    3.0% | ************************
    5.0% | ********
   10.0% | *

[DECISION VERDICT]: Bayesian filter successfully contained the 3% flaky test in
quarantine for 98.5% of its lifecycle. The naive streak rule leaked it into
blocking CI 73.5% of the time!
================================================================================
```



## Part 5: Naive AI Prompt vs. Calibrated Epistemic Specification

| Architectural Feature | Naive AI Prompting ((10, 2) Streak) | Calibrated Epistemic Specification (Bayesian Filter) |
| :--- | :--- | :--- |
| **Telemetry History** | Amnesiac: remembers only last 10 runs | Smooth recency: exponential memory ($\gamma = 0.98, N_{\text{eff}} = 50$) |
| **Survivorship Resistance** | Fails completely: $73.5\%$ false promotion rate | Robust: $<2\%$ false promotion on $3\%$ flaky tests |
| **Response to Degradation** | Sluggish: requires consecutive failures | Instant: single failure collapses posterior mass |
| **Recovery Mechanism** | Any lucky 10-pass streak resets gate | Requires $\approx 54$ clean runs to rebuild $95\%$ confidence |
| **Statistical Meaning** | Arbitrary rule of thumb | True posterior probability $P(\theta \le 0.010 \mid \mathcal{D}) \ge 0.95$ |
| **Computational Footprint** | Integer counter | Two floats $(\alpha_t, \beta_t)$; $\mathcal{O}(1)$ memory and compute |



## Part 6: 🎯 Check Your Understanding — 3-Tier "Audit the AI" Exercises

### 🟢 Tier 1: Spot the AI Hallucination
An AI assistant proposes a test promotion rule:
> *"To ensure maximum quality, we require a test to pass 20 times in a row
> before promotion. Because (1 - 0.05)^20 = 35.8%, a 5% flaky test has only a
> 35.8% chance of getting promoted. This guarantees that 64.2% of broken tests
> are permanently blocked."*

**Your Task**: Explain why the AI's calculation commits the Gambler's Fallacy
and calculate the true probability of a $5\%$ flaky test getting promoted if it
runs 200 times in staging.

<details><summary><b>Click here to view the solution...</b></summary>

**Solution**:
1. **The Fallacy**: The AI calculates the probability of a 20-streak on a
   *single, isolated attempt* of 20 runs. But tests in quarantine run
   continuously across dozens of PRs. Staging tests are not tested once; they
   are tested in a rolling stream!
2. **True Probability over 200 Runs**:
   Over $N = 200$ runs with $p = 0.05$, the probability of achieving *at least
one* streak of 20 consecutive passes is approximately:
   $$P(\text{Streak } \ge 20 \text{ in } 200 \text{ runs}) \approx 1 - (1 - 0.3584)^{200 / 20} \approx 1 - (0.6416)^{10} \approx \mathbf{98.8\%}!$$
   In reality, nearly $99\%$ of broken $5\%$ flaky tests will achieve a 20-pass
streak and get promoted!
</details>

---

### 🟡 Tier 2: Refactor the Flawed Specification
An engineering team uses a rolling 30-day window to evaluate test flakiness:
`flakiness = failures_last_30_days / runs_last_30_days`.
Explain the "cliff-edge problem" of rolling windows and rewrite the
specification into an exponential decay filter with an equivalent 30-day
half-life assuming 5 builds per day.

<details><summary><b>Click here to view the solution...</b></summary>

**Analysis & Refactored Specification**:
* **The Cliff-Edge Problem**: In a strict 30-day rolling window, a failure
  that occurred on Day 1 remains at full weight for 30 days, and then abruptly
  drops off a cliff on Day 31. This causes artificial step-function jumps in
  test scores.
* **Calibrating Half-Life**:
  At 5 builds per day, a 30-day half-life equals $T_{1/2} = 150 \text{ builds}$.
  $$\gamma = e^{-\ln(2) / T_{1/2}} = e^{-0.69315 / 150} \approx \mathbf{0.99539}$$
* **Declarative Specification**:
```yaml
decay_filter:
  half_life_builds: 150
  gamma: 0.99539
  n_eff: 216.9  # 1 / (1 - gamma)
  update_step:
    on_build:
      alpha: "0.99539 * alpha + (1.0 if failed else 0.0)"
      beta: "0.99539 * beta + (1.0 if passed else 0.0)"
  metric: "alpha / (alpha + beta)"
```
</details>

---

### 🔴 Tier 3: Architect an Autonomous Agent Contract
Design an autonomous gatekeeper agent that manages three distinct tiers:
1. **Tier 1 (Quarantine)**: Non-blocking, runs on all builds.
2. **Tier 2 (Candidate)**: Non-blocking, requires $P(\theta \le 0.02) \ge 0.90$.
3. **Tier 3 (Blocking CI)**: Blocking gate, requires $P(\theta \le 0.005) \ge
   0.95$.
Specify the exact mathematical transition rules and demotion criteria.

<details><summary><b>Click here to view the solution...</b></summary>

**Multi-Tier Agent Contract**:
```yaml
three_tier_gatekeeper:
  memory_decay: 0.98
  tier_definitions:
    quarantine:
      description: "Default landing tier for new or repaired tests"
      blocking: false
      promotion_rule: "Promote to Candidate if P(theta <= 0.02 | Telemetry) >= 0.90"
      
    candidate:
      description: "Probationary tier; runs on all PRs but non-blocking"
      blocking: false
      promotion_rule: "Promote to Blocking if P(theta <= 0.005 | Telemetry) >= 0.95"
      demotion_rule: "Demote to Quarantine if P(theta <= 0.02 | Telemetry) < 0.80"
      
    blocking:
      description: "Official release gatekeeper; failure halts deployment"
      blocking: true
      demotion_rule: "Demote to Candidate immediately if P(theta <= 0.005 | Telemetry) < 0.90"
      emergency_demotion: "Demote directly to Quarantine if single build failure causes P(theta > 0.03) >= 0.50"
```
</details>



## Key Takeaways & Course Synthesis

1. **Streak Heuristics Suffer from Survivorship Bias**: A $5\%$ flaky test has a
   $58.4\%$ probability of achieving a 50-pass streak across 500 builds. Never
   trust streak rules to promote tests.
2. **Structural Amnesia Ignores Context**: Streak rules treat a test with 10
   fails followed by 10 passes identically to a test with 1,000 clean runs.
3. **Bayesian Dynamic Filters Provide $O(1)$ Memory**: Exponential memory
   decay ($\gamma = 0.98$) requires storing only two numbers ($\alpha_t,
   \beta_t$) while maintaining full historical evidence weighted by recency.
4. **Immediate Demotion Protects Developers**: A single failure under the
   Bayesian filter immediately collapses posterior confidence, demoting unstable
   tests before they disrupt engineering velocity.

---

## 🧭 Course Navigation

* 🏠 **[Course Home & Master Syllabus](00_START_HERE.ipynb)**
* ⬅️ **Previous**: **[Ch 19: Verifying Flaky Fixes (SPRT & BF)](19_verifying_flaky_test_fixes.ipynb)** *(Part V: Production Reliability, CI/CD & Active Learning)*
* ➡️ **Next**: **[Ch 21: Framework Reliability Contracts](21_framework_reliability_and_tolerance_contracts.ipynb)** *(Part V: Production Reliability, CI/CD & Active Learning)*
* 🔄 **Companion Tracks for Chapter 20**: [📖 Conceptual](../conceptual/20_test_promotion_heuristics_vs_bayes.ipynb) | [🤖 AI-Native Track](README.md) | [🐍 Python](../python/20_test_promotion_heuristics_vs_bayes.ipynb) | [🔵 R](../r/20_test_promotion_heuristics_vs_bayes.ipynb)

